# Point Cloud to SurfaceRZFourier Fitting

Given a point cloud (xyz) from an unknown SurfaceRZFourier, reconstruct the surface.
See the accompanying `README.md` for a detailed description of the approach.

In [ ]:
import json
import pathlib

import numpy as np
import matplotlib.pyplot as plt

from constellaration.geometry import (
    surface_rz_fourier,
    surface_utils,
)
from constellaration.geometry.spectral_condensation import (
    SpectralCondensationSettings,
    spectrally_condense_surface,
)

# Repo root is two levels up from notebooks/point_cloud_fitting/
if "__file__" in dir():
    _REPO_ROOT = (
        pathlib.Path(__file__).resolve().parent.parent.parent
    )
else:
    _REPO_ROOT = pathlib.Path.cwd().parent.parent
TEST_DATA_DIR = _REPO_ROOT / "tests" / "geometry" / "test_data"
if not TEST_DATA_DIR.exists():
    TEST_DATA_DIR = pathlib.Path(
        "../../tests/geometry/test_data"
    ).resolve()

%matplotlib inline

In [ ]:
def plot_surface_3d(
    surface, title="Surface",
    n_theta=64, n_phi=64, ax=None, color=None,
):
    """Plot a SurfaceRZFourier as a 3D surface."""
    phi_ub = 2 * np.pi / surface.n_field_periods
    theta_phi = surface_utils.make_theta_phi_grid(
        n_theta, n_phi,
        phi_upper_bound=phi_ub,
        include_endpoints=True,
    )
    xyz = np.asarray(
        surface_rz_fourier.evaluate_points_xyz(surface, theta_phi)
    )
    if ax is None:
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')
    kwargs = dict(alpha=0.6, edgecolor='none')
    if color is not None:
        kwargs['color'] = color
    ax.plot_surface(
        xyz[:, :, 0], xyz[:, :, 1], xyz[:, :, 2], **kwargs
    )
    ax.set_title(title)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ranges = [
        xyz[:, :, i].max() - xyz[:, :, i].min()
        for i in range(3)
    ]
    max_range = np.array(ranges).max() / 2
    mid = np.array([xyz[:, :, i].mean() for i in range(3)])
    ax.set_xlim(mid[0] - max_range, mid[0] + max_range)
    ax.set_ylim(mid[1] - max_range, mid[1] + max_range)
    ax.set_zlim(mid[2] - max_range, mid[2] + max_range)
    return ax


def plot_rz_cross_section(
    surface, phi_val, label="", ax=None, **kwargs
):
    """Plot an R-Z cross-section at a given phi."""
    n_theta = 128
    theta = np.linspace(0, 2 * np.pi, n_theta, endpoint=True)
    theta_phi = np.stack(
        [theta, np.full_like(theta, phi_val)], axis=-1
    )
    rz = np.asarray(
        surface_rz_fourier.evaluate_points_rz(surface, theta_phi)
    )
    if ax is None:
        fig, ax = plt.subplots()
    ax.plot(rz[:, 0], rz[:, 1], label=label, **kwargs)
    ax.set_xlabel('R')
    ax.set_ylabel('Z')
    ax.set_aspect('equal')
    return ax

In [ ]:
def generate_point_cloud(
    surface, n_theta=32, n_phi=32,
    shuffle_theta=True, rng=None,
):
    """Generate a point cloud from a surface.

    Returns xyz points and their phi values. Simulates the
    scenario where we know the toroidal angle (phi) but not
    theta. Points at each phi slice are optionally shuffled
    to lose theta ordering.
    """
    phi_ub = 2 * np.pi / surface.n_field_periods
    theta_phi = surface_utils.make_theta_phi_grid(
        n_theta, n_phi, phi_upper_bound=phi_ub,
    )
    xyz = np.asarray(
        surface_rz_fourier.evaluate_points_xyz(
            surface, theta_phi,
        )
    )
    phis = theta_phi[0, :, 1]  # unique phi values

    if shuffle_theta and rng is not None:
        for j in range(n_phi):
            perm = rng.permutation(n_theta)
            xyz[:, j, :] = xyz[perm, j, :]

    return xyz, phis


def _sort_and_arclength_theta(xyz_slice):
    """Sort points by geometric angle, assign theta ~ arc length."""
    R = np.sqrt(xyz_slice[:, 0]**2 + xyz_slice[:, 1]**2)
    Z = xyz_slice[:, 2]
    angles = np.arctan2(Z - Z.mean(), R - R.mean())
    sort_idx = np.argsort(angles)
    sorted_xyz = xyz_slice[sort_idx]
    seg_lengths = np.sqrt(
        np.sum(np.diff(sorted_xyz, axis=0)**2, axis=-1)
    )
    close_len = np.sqrt(
        np.sum((sorted_xyz[0] - sorted_xyz[-1])**2)
    )
    cumulative = np.concatenate(
        [[0], np.cumsum(seg_lengths)]
    )
    total_length = cumulative[-1] + close_len
    thetas = cumulative / total_length * 2 * np.pi
    return sort_idx, thetas


def _project_points_rz(data_xyz, surface, phi_val, n_fine=2048):
    """For each data point, find the closest theta on the
    fitted curve in the R-Z plane. Uses vectorized
    line-segment projection.

    Returns theta values per point (NOT necessarily sorted).
    """
    fine_thetas = np.linspace(
        0, 2 * np.pi, n_fine, endpoint=False,
    )
    tp = np.stack(
        [fine_thetas, np.full(n_fine, phi_val)], axis=-1,
    )
    curve_rz = np.asarray(
        surface_rz_fourier.evaluate_points_rz(surface, tp)
    )

    R_data = np.sqrt(
        data_xyz[:, 0]**2 + data_xyz[:, 1]**2
    )
    Z_data = data_xyz[:, 2]
    data_rz = np.stack([R_data, Z_data], axis=-1)

    seg_starts = curve_rz
    seg_ends = np.roll(curve_rz, -1, axis=0)
    seg_d = seg_ends - seg_starts
    seg_d_sq = np.sum(seg_d**2, axis=-1)
    d_theta = 2 * np.pi / n_fine

    n_pts = len(data_rz)
    v = (
        data_rz[:, np.newaxis, :]
        - seg_starts[np.newaxis, :, :]
    )
    t_num = np.sum(v * seg_d[np.newaxis, :, :], axis=-1)
    safe_d_sq = np.where(seg_d_sq > 1e-30, seg_d_sq, 1.0)
    t = np.clip(t_num / safe_d_sq[np.newaxis, :], 0.0, 1.0)
    closest = (
        seg_starts[np.newaxis, :, :]
        + t[:, :, np.newaxis] * seg_d[np.newaxis, :, :]
    )
    dist_sq = np.sum(
        (data_rz[:, np.newaxis, :] - closest)**2, axis=-1,
    )
    best_seg = np.argmin(dist_sq, axis=1)
    t_best = t[np.arange(n_pts), best_seg]
    return (
        (fine_thetas[best_seg] + t_best * d_theta)
        % (2 * np.pi)
    )


def _fix_outlier_thetas(
    xyz_reordered, theta_phi, surface, phis,
    outlier_k=8.0, n_fine=2048,
):
    """Detect points with high R-Z residuals and reassign
    their theta using a local re-projection constrained to
    a window around the expected theta.
    """
    n_theta, n_phi = xyz_reordered.shape[:2]
    n_fixed = 0

    fine_thetas_full = np.linspace(
        0, 2 * np.pi, n_fine, endpoint=False,
    )
    d_theta = 2 * np.pi / n_fine

    for j in range(n_phi):
        tp_slice = theta_phi[:, j, :]
        fitted_rz = np.asarray(
            surface_rz_fourier.evaluate_points_rz(
                surface, tp_slice,
            )
        )
        R_data = np.sqrt(
            xyz_reordered[:, j, 0]**2
            + xyz_reordered[:, j, 1]**2
        )
        Z_data = xyz_reordered[:, j, 2]
        data_rz = np.stack([R_data, Z_data], axis=-1)
        residuals = np.sqrt(
            np.sum((data_rz - fitted_rz)**2, axis=-1)
        )

        median_res = np.median(residuals)
        if median_res < 1e-15:
            continue
        outlier_mask = residuals > outlier_k * median_res

        if not np.any(outlier_mask):
            continue

        good_idx = np.where(~outlier_mask)[0]
        bad_idx = np.where(outlier_mask)[0]
        if len(good_idx) < 3:
            continue

        tp_fine = np.stack(
            [fine_thetas_full, np.full(n_fine, phis[j])],
            axis=-1,
        )
        curve_rz = np.asarray(
            surface_rz_fourier.evaluate_points_rz(
                surface, tp_fine,
            )
        )

        for bi in bad_idx:
            dists_idx = np.abs(good_idx - bi)
            dists_idx = np.minimum(
                dists_idx, n_theta - dists_idx,
            )
            nearest_two = good_idx[
                np.argsort(dists_idx)[:2]
            ]
            t1 = theta_phi[nearest_two[0], j, 0]
            t2 = theta_phi[nearest_two[1], j, 0]
            diff = (t2 - t1 + np.pi) % (2 * np.pi) - np.pi
            expected = (t1 + diff / 2) % (2 * np.pi)
            half_gap = abs(diff) / 2 + d_theta
            lo = (expected - half_gap) % (2 * np.pi)
            hi = (expected + half_gap) % (2 * np.pi)
            if lo < hi:
                mask = (
                    (fine_thetas_full >= lo)
                    & (fine_thetas_full <= hi)
                )
            else:
                mask = (
                    (fine_thetas_full >= lo)
                    | (fine_thetas_full <= hi)
                )
            if not np.any(mask):
                theta_phi[bi, j, 0] = expected
                n_fixed += 1
                continue
            local_idx = np.where(mask)[0]
            local_curve = curve_rz[local_idx]
            pt_rz = data_rz[bi]
            dists_sq = np.sum(
                (local_curve - pt_rz[np.newaxis, :])**2,
                axis=-1,
            )
            best_local = local_idx[np.argmin(dists_sq)]
            theta_phi[bi, j, 0] = fine_thetas_full[
                best_local
            ]
            n_fixed += 1

    return n_fixed


def fit_surface_from_point_cloud(
    xyz, phis, n_field_periods,
    n_poloidal_modes=8, max_toroidal_mode=None,
    is_stellarator_symmetric=True,
    max_iterations=40, theta_tol=1e-5,
    relaxation=0.5, verbose=True,
    init_surface=None,
):
    """Fit a SurfaceRZFourier via alternating optimization.

    1. Initialize theta via arc-length parameterization
       (or project onto init_surface for coarse-to-fine).
    2. Alternate between:
       (a) Fit Fourier coefficients (linear least squares)
       (b) Update theta by R-Z projection + damping.
    3. Track the best solution by RSS.
    4. Fix outlier points and re-fit.
    """
    n_theta, n_phi = xyz.shape[0], xyz.shape[1]

    if max_toroidal_mode is None:
        max_toroidal_mode = max(1, n_phi // 4)

    n_toroidal_modes = 2 * max_toroidal_mode + 1

    # Step 0: Initialize theta
    theta_phi = np.zeros((n_theta, n_phi, 2))
    xyz_reordered = np.zeros_like(xyz)

    if init_surface is not None:
        for j in range(n_phi):
            sort_idx, _ = _sort_and_arclength_theta(
                xyz[:, j, :],
            )
            xyz_sorted = xyz[sort_idx, j, :]
            thetas = _project_points_rz(
                xyz_sorted, init_surface, phis[j],
            )
            order = np.argsort(thetas)
            theta_phi[:, j, 0] = thetas[order]
            theta_phi[:, j, 1] = phis[j]
            xyz_reordered[:, j, :] = xyz_sorted[order]
        if verbose:
            print("  Initialized theta from provided surface.")
    else:
        for j in range(n_phi):
            sort_idx, thetas = _sort_and_arclength_theta(
                xyz[:, j, :],
            )
            theta_phi[:, j, 0] = thetas
            theta_phi[:, j, 1] = phis[j]
            xyz_reordered[:, j, :] = xyz[sort_idx, j, :]

    best_rss = np.inf
    best_surface = None
    best_theta_phi = None
    best_xyz_reordered = None

    for iteration in range(max_iterations):
        # (a) Fit coefficients given current theta
        fitted_surface, residual = (
            surface_rz_fourier.from_points(
                points=xyz_reordered,
                theta_phi=theta_phi,
                n_field_periods=n_field_periods,
                n_poloidal_modes=n_poloidal_modes,
                n_toroidal_modes=n_toroidal_modes,
                is_stellarator_symmetric=is_stellarator_symmetric,
            )
        )

        # Track best by RSS
        fitted_xyz = np.asarray(
            surface_rz_fourier.evaluate_points_xyz(
                fitted_surface, theta_phi,
            )
        )
        rss = float(
            np.sum((xyz_reordered - fitted_xyz)**2)
        )
        if rss < best_rss:
            best_rss = rss
            best_surface = fitted_surface
            best_theta_phi = theta_phi.copy()
            best_xyz_reordered = xyz_reordered.copy()

        # (b) Update theta: R-Z projection + re-sort + damp
        max_theta_change = 0.0
        for j in range(n_phi):
            old_thetas = theta_phi[:, j, 0].copy()
            new_thetas = _project_points_rz(
                xyz_reordered[:, j, :],
                fitted_surface,
                phis[j],
            )
            new_sort = np.argsort(new_thetas)
            new_thetas_sorted = new_thetas[new_sort]
            delta = (
                (new_thetas_sorted - old_thetas + np.pi)
                % (2 * np.pi) - np.pi
            )
            blended = np.sort(
                (old_thetas + relaxation * delta)
                % (2 * np.pi)
            )
            theta_phi[:, j, 0] = blended
            xyz_reordered[:, j, :] = (
                xyz_reordered[new_sort, j, :]
            )
            max_theta_change = max(
                max_theta_change, np.max(np.abs(delta)),
            )

        if verbose:
            print(
                f"  Iter {iteration + 1}: "
                f"rss={rss:.4e}, "
                f"max_theta_change={max_theta_change:.2e}"
            )

        if max_theta_change < theta_tol:
            if verbose:
                print(
                    f"  Converged after "
                    f"{iteration + 1} iterations."
                )
            break

    # Step 3: Outlier correction
    theta_phi = best_theta_phi.copy()
    xyz_reordered = best_xyz_reordered.copy()
    n_fixed = _fix_outlier_thetas(
        xyz_reordered, theta_phi, best_surface, phis,
    )
    if n_fixed > 0:
        if verbose:
            print(
                f"  Fixed {n_fixed} outlier point(s), "
                f"re-fitting..."
            )
        for j in range(n_phi):
            sort_idx = np.argsort(theta_phi[:, j, 0])
            theta_phi[:, j, 0] = (
                theta_phi[sort_idx, j, 0]
            )
            xyz_reordered[:, j, :] = (
                xyz_reordered[sort_idx, j, :]
            )
        fixed_surface, _ = surface_rz_fourier.from_points(
            points=xyz_reordered,
            theta_phi=theta_phi,
            n_field_periods=n_field_periods,
            n_poloidal_modes=n_poloidal_modes,
            n_toroidal_modes=n_toroidal_modes,
            is_stellarator_symmetric=is_stellarator_symmetric,
        )
        fitted_xyz = np.asarray(
            surface_rz_fourier.evaluate_points_xyz(
                fixed_surface, theta_phi,
            )
        )
        fixed_rss = float(
            np.sum((xyz_reordered - fitted_xyz)**2)
        )
        if verbose:
            print(
                f"  After outlier fix: "
                f"rss={fixed_rss:.4e} "
                f"(was {best_rss:.4e})"
            )
        if fixed_rss < best_rss:
            best_surface = fixed_surface
            best_rss = fixed_rss
        elif verbose:
            print(
                "  Outlier fix did not improve RSS, "
                "keeping previous best."
            )

    return best_surface, best_rss


def evaluate_reconstruction(original, reconstructed, label=""):
    """Evaluate and visualize surface reconstruction quality."""
    n_pol, n_tor = (
        surface_utils
        .n_poloidal_toroidal_points_to_satisfy_nyquist_criterion(
            max(
                original.n_poloidal_modes,
                reconstructed.n_poloidal_modes,
            ),
            max(
                original.max_toroidal_mode,
                reconstructed.max_toroidal_mode,
            ),
        )
    )
    if (
        original.max_toroidal_mode == 0
        and reconstructed.max_toroidal_mode == 0
    ):
        n_tor = 1

    distances = np.asarray(
        surface_rz_fourier.compute_normal_displacement(
            original, reconstructed, n_pol, n_tor,
        )
    )

    print(f"--- {label} ---")
    print(
        f"  Max |normal displacement|: "
        f"{np.max(np.abs(distances)):.6e}"
    )
    print(
        f"  RMS normal displacement:   "
        f"{np.sqrt(np.mean(distances**2)):.6e}"
    )
    print(
        f"  Mean normal displacement:  "
        f"{np.mean(distances):.6e}"
    )

    # R-Z cross-section comparison
    phi_ub = 2 * np.pi / original.n_field_periods
    n_phi_plots = min(
        4, max(1, original.max_toroidal_mode + 1),
    )
    fig, axes = plt.subplots(
        1, n_phi_plots, figsize=(5 * n_phi_plots, 5),
    )
    if n_phi_plots == 1:
        axes = [axes]
    for i, ax in enumerate(axes):
        phi_val = i * phi_ub / max(n_phi_plots, 1)
        plot_rz_cross_section(
            original, phi_val, label="Original",
            ax=ax, color='blue', linewidth=2,
        )
        plot_rz_cross_section(
            reconstructed, phi_val,
            label="Reconstructed",
            ax=ax, color='red',
            linestyle='--', linewidth=2,
        )
        ax.set_title(f"phi = {phi_val:.2f}")
        ax.legend()
    plt.suptitle(label)
    plt.tight_layout()
    plt.show()

    return distances

## Test 1: D-Shape (axisymmetric, from Hirshman 1985)

In [ ]:
rng = np.random.default_rng(42)

# D-Shape from Hirshman 1985
d_shape = surface_rz_fourier.SurfaceRZFourier(
    r_cos=np.array([3.0, 0.991, 0.136]).reshape(-1, 1),
    z_sin=np.array([0.0, 1.409, -0.118]).reshape(-1, 1),
    n_field_periods=1,
    is_stellarator_symmetric=True,
)

# Generate point cloud (shuffled theta ordering)
xyz_cloud, phis = generate_point_cloud(
    d_shape, n_theta=64, n_phi=1,
    shuffle_theta=True, rng=rng,
)
print(f"Point cloud shape: {xyz_cloud.shape}")

# Fit with alternating optimization
fitted, rss = fit_surface_from_point_cloud(
    xyz_cloud, phis, n_field_periods=1,
    n_poloidal_modes=8, max_toroidal_mode=0,
)

# Spectral condensation
settings = SpectralCondensationSettings(
    p=4, q=1, normalize=False,
    maximum_normal_displacement=1e-4, n_restarts=1,
)
condensed = spectrally_condense_surface(fitted, settings)

# Evaluate
print("\n=== Fitted (before condensation) ===")
_ = evaluate_reconstruction(
    d_shape, fitted, label="D-Shape: Fitted",
)

print("\n=== After spectral condensation ===")
_ = evaluate_reconstruction(
    d_shape, condensed, label="D-Shape: Condensed",
)

surfs = [
    ("Original", d_shape),
    ("Fitted", fitted),
    ("Condensed", condensed),
]
for name, surf in surfs:
    sw = float(surface_rz_fourier.spectral_width(
        [surf.r_cos, surf.z_sin],
        p=4, q=1, normalize=True,
    ))
    print(f"  {name:12s} spectral width: {sw:.4f}")

## Test 2: 3D stellarator boundary from test data

In [ ]:
# Load a 3D stellarator boundary from the test data
with (TEST_DATA_DIR / "test_boundary_1.json").open() as f:
    data = json.load(f)

original_3d = surface_rz_fourier.SurfaceRZFourier(
    r_cos=np.array(data["r_cos"]),
    z_sin=np.array(data["z_sin"]),
    n_field_periods=data["n_field_periods"],
    is_stellarator_symmetric=data["is_stellarator_symmetric"],
)
print(
    f"Original surface: "
    f"mpol={original_3d.max_poloidal_mode}, "
    f"ntor={original_3d.max_toroidal_mode}, "
    f"nfp={original_3d.n_field_periods}"
)

# Generate point cloud (shuffled)
xyz_cloud_3d, phis_3d = generate_point_cloud(
    original_3d, n_theta=64, n_phi=32,
    shuffle_theta=True, rng=rng,
)
print(f"Point cloud shape: {xyz_cloud_3d.shape}")

# Fit with alternating optimization
fitted_3d, rss_3d = fit_surface_from_point_cloud(
    xyz_cloud_3d, phis_3d,
    n_field_periods=original_3d.n_field_periods,
    n_poloidal_modes=8, max_toroidal_mode=6,
)

# Spectral condensation
settings_3d = SpectralCondensationSettings(
    p=4, q=1, normalize=False,
    maximum_normal_displacement=1e-3, n_restarts=1,
)
condensed_3d = spectrally_condense_surface(
    fitted_3d, settings_3d,
)

# Evaluate
print("\n=== Fitted (before condensation) ===")
_ = evaluate_reconstruction(
    original_3d, fitted_3d,
    label="3D Stellarator: Fitted",
)

print("\n=== After spectral condensation ===")
_ = evaluate_reconstruction(
    original_3d, condensed_3d,
    label="3D Stellarator: Condensed",
)

surfs = [
    ("Original", original_3d),
    ("Fitted", fitted_3d),
    ("Condensed", condensed_3d),
]
for name, surf in surfs:
    sw = float(surface_rz_fourier.spectral_width(
        [surf.r_cos, surf.z_sin],
        p=4, q=1, normalize=True,
    ))
    print(f"  {name:12s} spectral width: {sw:.4f}")

## Test 3: Boundary from the constellaration HuggingFace dataset

In [ ]:
import datasets

# Load one boundary from the HuggingFace dataset
ds = datasets.load_dataset(
    "proxima-fusion/constellaration", "default",
    split="train",
)
row = ds[0]

original_hf = (
    surface_rz_fourier.SurfaceRZFourier
    .model_validate_json(row["boundary.json"])
)
print(
    f"HF surface: "
    f"mpol={original_hf.max_poloidal_mode}, "
    f"ntor={original_hf.max_toroidal_mode}, "
    f"nfp={original_hf.n_field_periods}"
)

# Generate point cloud (shuffled)
xyz_cloud_hf, phis_hf = generate_point_cloud(
    original_hf, n_theta=64, n_phi=32,
    shuffle_theta=True, rng=rng,
)
print(f"Point cloud shape: {xyz_cloud_hf.shape}")

# Coarse-to-fine: first fit with low resolution to get
# robust theta, then refine at the original's resolution.
print("--- Phase 1: coarse fit (mpol=3, ntor=3) ---")
fitted_coarse, _ = fit_surface_from_point_cloud(
    xyz_cloud_hf, phis_hf,
    n_field_periods=original_hf.n_field_periods,
    n_poloidal_modes=4, max_toroidal_mode=3,
    max_iterations=40,
)

print(
    "\n--- Phase 2: fine fit at original resolution "
    "(mpol=4, ntor=4) ---"
)
fitted_hf, rss_hf = fit_surface_from_point_cloud(
    xyz_cloud_hf, phis_hf,
    n_field_periods=original_hf.n_field_periods,
    n_poloidal_modes=original_hf.n_poloidal_modes,
    max_toroidal_mode=original_hf.max_toroidal_mode,
    max_iterations=80,
    init_surface=fitted_coarse,
)

# Spectral condensation
settings_hf = SpectralCondensationSettings(
    p=4, q=1, normalize=False,
    maximum_normal_displacement=1e-3, n_restarts=1,
)
condensed_hf = spectrally_condense_surface(
    fitted_hf, settings_hf,
)

# Evaluate
print("\n=== Fitted (before condensation) ===")
_ = evaluate_reconstruction(
    original_hf, fitted_hf,
    label="HuggingFace: Fitted",
)

print("\n=== After spectral condensation ===")
_ = evaluate_reconstruction(
    original_hf, condensed_hf,
    label="HuggingFace: Condensed",
)

surfs = [
    ("Original", original_hf),
    ("Fitted", fitted_hf),
    ("Condensed", condensed_hf),
]
for name, surf in surfs:
    sw = float(surface_rz_fourier.spectral_width(
        [surf.r_cos, surf.z_sin],
        p=4, q=1, normalize=True,
    ))
    print(f"  {name:12s} spectral width: {sw:.4f}")

In [ ]:
import datasets

# Load another boundary from the HuggingFace dataset
ds = datasets.load_dataset(
    "proxima-fusion/constellaration", "default",
    split="train",
)
row = ds[7]

original_hf = (
    surface_rz_fourier.SurfaceRZFourier
    .model_validate_json(row["boundary.json"])
)
print(
    f"HF surface: "
    f"mpol={original_hf.max_poloidal_mode}, "
    f"ntor={original_hf.max_toroidal_mode}, "
    f"nfp={original_hf.n_field_periods}"
)

# Generate point cloud (shuffled)
xyz_cloud_hf, phis_hf = generate_point_cloud(
    original_hf, n_theta=64, n_phi=32,
    shuffle_theta=True, rng=rng,
)
print(f"Point cloud shape: {xyz_cloud_hf.shape}")

# Coarse-to-fine
print("--- Phase 1: coarse fit (mpol=3, ntor=3) ---")
fitted_coarse, _ = fit_surface_from_point_cloud(
    xyz_cloud_hf, phis_hf,
    n_field_periods=original_hf.n_field_periods,
    n_poloidal_modes=4, max_toroidal_mode=3,
    max_iterations=40,
)

print(
    "\n--- Phase 2: fine fit at original resolution "
    "(mpol=4, ntor=4) ---"
)
fitted_hf, rss_hf = fit_surface_from_point_cloud(
    xyz_cloud_hf, phis_hf,
    n_field_periods=original_hf.n_field_periods,
    n_poloidal_modes=original_hf.n_poloidal_modes,
    max_toroidal_mode=original_hf.max_toroidal_mode,
    max_iterations=80,
    init_surface=fitted_coarse,
)

# Spectral condensation
settings_hf = SpectralCondensationSettings(
    p=4, q=1, normalize=False,
    maximum_normal_displacement=1e-3, n_restarts=1,
)
condensed_hf = spectrally_condense_surface(
    fitted_hf, settings_hf,
)

# Evaluate
print("\n=== Fitted (before condensation) ===")
_ = evaluate_reconstruction(
    original_hf, fitted_hf,
    label="HuggingFace: Fitted",
)

print("\n=== After spectral condensation ===")
_ = evaluate_reconstruction(
    original_hf, condensed_hf,
    label="HuggingFace: Condensed",
)

surfs = [
    ("Original", original_hf),
    ("Fitted", fitted_hf),
    ("Condensed", condensed_hf),
]
for name, surf in surfs:
    sw = float(surface_rz_fourier.spectral_width(
        [surf.r_cos, surf.z_sin],
        p=4, q=1, normalize=True,
    ))
    print(f"  {name:12s} spectral width: {sw:.4f}")